# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{getattr(meta, 'name', None)}: {getattr(meta, 'description', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)} | Version: {getattr(meta, 'version', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id in the dataset
record_sets = []
for rs in getattr(meta, 'recordSet', []):
    rs_id = getattr(rs, '@id', None)
    name = getattr(rs, 'name', None)
    print(f"Record set: {name} (ID: {rs_id})")
    record_sets.append(rs_id)
    # List the fields within this record set
    print("  Fields:")
    for field in getattr(rs, 'field', []):
        field_id = getattr(field, '@id', None)
        fname = getattr(field, 'name', None)
        dtype = getattr(field, 'dataType', None)
        print(f"    - {fname} (@id: {field_id}, type: {dtype})")
    print()
# For the FAIR^2 dataset, get first record set id for further referencing
if record_sets:
    first_record_set_id = record_sets[0]
else:
    # fallback if no record sets
    first_record_set_id = None

In [ ]:
# Optionally, see examples of records for the first available record set
if first_record_set_id:
    print(f"Sample records from record set {first_record_set_id}:")
    for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record sets into DataFrames (using @id references)
import warnings
warnings.filterwarnings('ignore')

dataframes = {}
for rec_id in record_sets:
    recs = list(dataset.records(record_set=rec_id))
    if recs:
        dataframes[rec_id] = pd.DataFrame(recs)
        print(f"Loaded {len(dataframes[rec_id])} records from {rec_id}.")
    else:
        print(f"No records found for record set {rec_id}.")
# Print sample columns for first record set
if first_record_set_id and first_record_set_id in dataframes:
    print("Columns:", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# Identify a numeric field from the first record set (if available)
if first_record_set_id and first_record_set_id in dataframes:
    df = dataframes[first_record_set_id]
    # Try to auto-detect a numeric column
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    # If not found, try manual fallback
    if numeric_field is None:
        # Common protein quantification columns
        options = [c for c in df.columns if any(x in c.lower() for x in ['coverage', 'count', 'mw', 'pi', 'abundance', 'intensity'])]
        if options:
            numeric_field = options[0]
    print(f"Example numeric field for analysis: {numeric_field}")
    # Apply EDA: Filter, normalize, group
    threshold = None
    if numeric_field is not None:
        # Try to make sure values are float
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Drop missing values in this field
        filter_mask = df[numeric_field].notnull()
        df = df[filter_mask]
        if not df.empty:
            # Use 25th percentile as example threshold
            threshold = df[numeric_field].quantile(0.25)
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())
            # Normalization
            mean = filtered_df[numeric_field].mean()
            std = filtered_df[numeric_field].std()
            filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - mean) / std
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
            # Try to group by another column (example: accession/protein/family/sample)
            group_candidates = [c for c in filtered_df.columns if any(x in c.lower() for x in ['accession', 'sample', 'family', 'condition'])]
            if group_candidates:
                group_field = group_candidates[0]
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped by {group_field}: (mean)")
                display(grouped_df.head())
            else:
                group_field = None
        else:
            print(f"No non-null records to filter on field {numeric_field}.")
    else:
        print("No numeric field found for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if first_record_set_id and first_record_set_id in dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field].dropna(), bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()
    if group_candidates:
        # Boxplot by group
        grouped_data = df.groupby(group_candidates[0])[numeric_field].mean().sort_values(ascending=False)
        grouped_data.head(10).plot(kind='bar', figsize=(10, 4))
        plt.title(f"Mean {numeric_field} by {group_candidates[0]} (top 10 groups)")
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we demonstrated how to load and explore the FAIR^2 mass spectrometry dataset using `mlcroissant`, referencing all entities by their `@id` where applicable.
* We reviewed dataset structure, examined available record sets, fields, and extracted a DataFrame for further analysis.
* Exploratory data analysis (EDA) showed how to filter and normalize numeric fields, with dynamic selection of field names using IDs, and visualized distributions to uncover variation within the dataset.
* Such analysis is useful in mass spectrometry datasets for comparing protein abundance, identifying outliers, and preparing the data for scientific or machine learning workflows.

For further tasks, one might build predictive models, perform supervised learning, or conduct deeper protein annotation analysis based on the prepared DataFrames.